In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install sentence-transformers faiss-cpu transformers -q

import faiss
import pickle
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import torch
import warnings
warnings.filterwarnings('ignore')

print("Setup complete.")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete.
Using device: cpu


In [ ]:
save_path = '/content/drive/MyDrive/rag-complaint-chatbot/vector_store/'

index = faiss.read_index(save_path + 'index.faiss')
print(f"Loaded index with {index.ntotal:,} vectors")

with open(save_path + 'chunks.pkl', 'rb') as f:
    chunked_data = pickle.load(f)
print(f"Loaded {len(chunked_data)} chunks")

embed_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
print("Embedding model loaded.")

Loaded index with 5,291 vectors
Loaded 5291 chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.


In [ ]:
def retrieve(query, k=5):
    query_emb = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    distances, indices = index.search(query_emb, k)
    results = []
    for i, idx in enumerate(indices[0]):
        if idx < len(chunked_data):
            results.append({
                'score': float(distances[0][i]),
                'product': chunked_data[idx]['product_category'],
                'text': chunked_data[idx]['chunk_text']
            })
    return results

print("Retriever function defined.")

Retriever function defined.


In [ ]:
test_query = "Why are customers unhappy with credit cards?"
results = retrieve(test_query, k=3)

print(f"Query: {test_query}")
print("\nTop 3 retrieved chunks:")
for i, r in enumerate(results):
    print(f"\n  [{i+1}] Score: {r['score']:.4f}")
    print(f"      Product: {r['product']}")
    print(f"      Text: {r['text'][:200]}...")

Query: Why are customers unhappy with credit cards?

Top 3 retrieved chunks:

  [1] Score: 0.5885
      Product: Credit card or prepaid card
      Text: he bad credit debt to all 3 major credit bureau and is currently reflecting on my credit report of payment owed of 85.00 and negative payment for 60 days. please explain why secured card customer don'...

  [2] Score: 0.5822
      Product: Credit card
      Text: t had any of the cards go into default, and they continue to send me offers to bring over my balances and investments that i have with other institutions to their bank. if they had let me know the car...

  [3] Score: 0.5695
      Product: Credit card
      Text: y, or the postal office, or another reputable bank in my town, all offers we denied. i feel these practices are predatory for seniors and people in rural areas. again, i haven't had this trouble with ...


In [ ]:
print("Loading LLM...")

try:
    generator = pipeline(
        "text-generation",
        model="google/flan-t5-base",
        device=0 if device == 'cuda' else -1,
        max_new_tokens=150
    )
    print("LLM (Flan-T5-base) loaded successfully.")

except Exception as e:
    print(f"Error with Flan-T5-base: {e}")
    print("Trying Flan-T5-small...")

    generator = pipeline(
        "text-generation",
        model="google/flan-t5-small",
        device=0 if device == 'cuda' else -1,
        max_new_tokens=100
    )
    print("LLM (Flan-T5-small) loaded successfully.")

Loading LLM...


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', '

LLM (Flan-T5-base) loaded successfully.


In [ ]:
def rag_answer(question, k=5):
    retrieved = retrieve(question, k)
    if not retrieved:
        return "No relevant information found.", []

    context = "\n\n".join([f"Excerpt {i+1}: {r['text']}" for i, r in enumerate(retrieved)])

    prompt = f"""You are a financial analyst assistant for CrediTrust. Your task is to answer questions about customer complaints. Use the following retrieved complaint excerpts to formulate your answer. If the context doesn't contain the answer, state that you don't have enough information.

Context:
{context}

Question: {question}

Answer:"""

    if callable(generator):
        answer = generator(prompt)
    else:
        answer = generator(prompt, do_sample=False)[0]['generated_text']

    return answer, retrieved

print("RAG pipeline defined.")

RAG pipeline defined.


In [ ]:
test_questions = [
    "Why are customers unhappy with credit cards?",
    "What are the main issues with money transfers?",
    "How do customers feel about personal loans?",
]

for q in test_questions:
    print("\n" + "="*60)
    print(f"Question: {q}")
    print("="*60)
    ans, ctx = rag_answer(q, k=3)
    print(f"\nAnswer: {ans}")
    print("\nRetrieved sources:")
    for i, r in enumerate(ctx):
        print(f"  [{i+1}] Score={r['score']:.3f} | Product={r['product']}")
        print(f"      Text: {r['text'][:120]}...")


Question: Why are customers unhappy with credit cards?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer: [{'generated_text': "You are a financial analyst assistant for CrediTrust. Your task is to answer questions about customer complaints. Use the following retrieved complaint excerpts to formulate your answer. If the context doesn't contain the answer, state that you don't have enough information.\n\nContext:\nExcerpt 1: he bad credit debt to all 3 major credit bureau and is currently reflecting on my credit report of payment owed of 85.00 and negative payment for 60 days. please explain why secured card customer don't have same services as regular credit card customers. we are just as important as we are trying to reestablish for economically, to start business and purchased homes for are family. us bank secured card is a way for the bank to make money off fees and annual memberships\n\nExcerpt 2: t had any of the cards go into default, and they continue to send me offers to bring over my balances and investments that i have with other institutions to their bank. if they had le

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer: [{'generated_text': "You are a financial analyst assistant for CrediTrust. Your task is to answer questions about customer complaints. Use the following retrieved complaint excerpts to formulate your answer. If the context doesn't contain the answer, state that you don't have enough information.\n\nContext:\nExcerpt 1: availability of funds and have no eta. there should never be more than a days worth of a wait to have access to your funds. it's unacceptable and xxxx should be fined and pay for every single fee we have for late payments, bounced payments, etc. the stress and anxiety caused by their lack of communication and broken system is going to get people hurt or killed... unacceptable to be messing with peoples money and making interest on the funds while we sit around and wait for them.\n\nExcerpt 2: ization. i never removed the funds into my bank account and sent the funds back t my friend via cashapp. they have basically stolen my money in order to reimburse my friend

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer: [{'generated_text': "You are a financial analyst assistant for CrediTrust. Your task is to answer questions about customer complaints. Use the following retrieved complaint excerpts to formulate your answer. If the context doesn't contain the answer, state that you don't have enough information.\n\nContext:\nExcerpt 1: i have been unemployed for more than 10 months and have been in continuous contact with the store about my financial situation and they have sent emails saying they are willing to work with borrowers and this is far from the truth. i borrowed 370.00 and have already paid that plus additional money and can't get caught up on payments due to the current job market and being unable to gain employment.\n\nExcerpt 2: one main financial in xxxx, al tricked their consumers to sign up for optional products and they were telling borrows that they could not get approved for a loan unless they signed up for these additional services. i was a customer from xxxx.\n\nExcerpt 

In [ ]:
print("="*60)
print("RAG EVALUATION")
print("="*60)

test_questions = [
    "Why are customers unhappy with credit cards?",
    "What are the main issues with money transfers?",
    "How do customers feel about personal loans?",
    "What complaints about savings accounts appear most often?",
    "Are there recurring problems with hidden fees?",
    "What are the most common complaints about Credit cards?",
    "How do customers describe their experience with money transfers?"
]

results = []
for q in test_questions:
    print(f"\nQuestion: {q}")
    ans, ctx = rag_answer(q, k=3)
    print(f"Answer: {ans}")
    print("Retrieved sources:")
    for i, r in enumerate(ctx):
        print(f"  [{i+1}] Score={r['score']:.3f} | Product={r['product']} | Text={r['text'][:80]}...")
    results.append({'question': q, 'answer': ans, 'sources': ctx})

RAG EVALUATION

Question: Why are customers unhappy with credit cards?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: [{'generated_text': "You are a financial analyst assistant for CrediTrust. Your task is to answer questions about customer complaints. Use the following retrieved complaint excerpts to formulate your answer. If the context doesn't contain the answer, state that you don't have enough information.\n\nContext:\nExcerpt 1: he bad credit debt to all 3 major credit bureau and is currently reflecting on my credit report of payment owed of 85.00 and negative payment for 60 days. please explain why secured card customer don't have same services as regular credit card customers. we are just as important as we are trying to reestablish for economically, to start business and purchased homes for are family. us bank secured card is a way for the bank to make money off fees and annual memberships\n\nExcerpt 2: t had any of the cards go into default, and they continue to send me offers to bring over my balances and investments that i have with other institutions to their bank. if they had let

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: [{'generated_text': "You are a financial analyst assistant for CrediTrust. Your task is to answer questions about customer complaints. Use the following retrieved complaint excerpts to formulate your answer. If the context doesn't contain the answer, state that you don't have enough information.\n\nContext:\nExcerpt 1: availability of funds and have no eta. there should never be more than a days worth of a wait to have access to your funds. it's unacceptable and xxxx should be fined and pay for every single fee we have for late payments, bounced payments, etc. the stress and anxiety caused by their lack of communication and broken system is going to get people hurt or killed... unacceptable to be messing with peoples money and making interest on the funds while we sit around and wait for them.\n\nExcerpt 2: ization. i never removed the funds into my bank account and sent the funds back t my friend via cashapp. they have basically stolen my money in order to reimburse my friend 

...

Answer: [{'generated_text': "You are a financial analyst assistant for CrediTrust. Your task is to answer questions about customer complaints. Use the following retrieved complaint excerpts to formulate your answer. If the context doesn't contain the answer, state that you don't have enough information.\n\nContext:\nExcerpt 1: constantly charged fees\n\nExcerpt 2: there were not supposed to be any fees. also, i had a thing on my account that was supposed to stop me from overdrafting and it did not work. it was a while ago so i don't remember fees and amounts.\n\nExcerpt 3: change the account back to the original product, which would eliminate the fee, but i was only eligible for one fee reversal. i feel that this is an unacceptable deceptive practice because i was charged 18.00 per month for the last few years with no explanation, no apology, and no restitution. the account was converted without my knowledge and, until today, i couldn't even get them to change it back to save future fe

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: [{'generated_text': "You are a financial analyst assistant for CrediTrust. Your task is to answer questions about customer complaints. Use the following retrieved complaint excerpts to formulate your answer. If the context doesn't contain the answer, state that you don't have enough information.\n\nContext:\nExcerpt 1: this complaint is for card ending in xxxx and is different than other complaints filed with the cfpb against bank of america i am writing to lodge a formal complaint regarding my recent experiences with bank of america. i have been a bank of america credit card holder for several years, and i have always strived to keep my account in good standing. unfortunately, my financial situation has been deteriorating quickly, and all my hopes and attempts at holding my finances together are exhausted.\n\nExcerpt 2: actions with the credit cards companies i used and today i talked to american express customer service and after talking with many representatives they told me

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: [{'generated_text': 'You are a financial analyst assistant for CrediTrust. Your task is to answer questions about customer complaints. Use the following retrieved complaint excerpts to formulate your answer. If the context doesn\'t contain the answer, state that you don\'t have enough information.\n\nContext:\nExcerpt 1: e but instead transferred " successfully. \'\' this needs to be addressed. if there is a law protecting this, it needs to be changed. taking from people when there was no real reason by the customer to be hit with a fee is a criminal act and no law should protect it. how many hard working americans who don\'t read the fine print or don\'t have the time to look deep into these unjust fees are being robbed daily? and how much of this unjust money is being put into these bankers pockets needs to stop.\n\nExcerpt 2: ization. i never removed the funds into my bank account and sent the funds back t my friend via cashapp. they have basically stolen my money in order t

In [ ]:
print("\n" + "="*60)
print("EVALUATION TABLE")
print("="*60)
print("| Question | Answer | Retrieved Sources | Quality Score (1-5) | Comments |")
print("|----------|--------|-------------------|---------------------|----------|")
for r in results:
    sources = ", ".join([f"{ctx['product']} ({ctx['score']:.2f})" for ctx in r['sources'][:2]])
    print(f"| {r['question'][:25]}... | {r['answer'][:25]}... | {sources} | ? | ... |")

print("\nPlease assign quality scores (1-5) based on relevance, accuracy, and completeness.")


EVALUATION TABLE
| Question | Answer | Retrieved Sources | Quality Score (1-5) | Comments |
|----------|--------|-------------------|---------------------|----------|
| Why are customers unhappy... | [{'generated_text': "You are a financial analyst assistant for CrediTrust. Your task is to answer questions about customer complaints. Use the following retrieved complaint excerpts to formulate your answer. If the context doesn't contain the answer, state that you don't have enough information.\n\nContext:\nExcerpt 1: he bad credit debt to all 3 major credit bureau and is currently reflecting on my credit report of payment owed of 85.00 and negative payment for 60 days. please explain why secured card customer don't have same services as regular credit card customers. we are just as important as we are trying to reestablish for economically, to start business and purchased homes for are family. us bank secured card is a way for the bank to make money off fees and annual memberships\n\nEx

In [ ]:
results_df = pd.DataFrame([{
    'question': r['question'],
    'answer': r['answer'],
    'sources': str([{'product': s['product'], 'score': s['score']} for s in r['sources']])
} for r in results])

results_df.to_csv('/content/drive/MyDrive/rag-complaint-chatbot/rag_evaluation_results.csv', index=False)
print("Results saved to: /content/drive/MyDrive/rag-complaint-chatbot/rag_evaluation_results.csv")

Results saved to: /content/drive/MyDrive/rag-complaint-chatbot/rag_evaluation_results.csv


In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/rag-complaint-chatbot/rag_evaluation_results.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
# ============================================================
# TASK 3: COMPLETE RAG PIPELINE WITH EVALUATION
# All code in one cell – run everything at once
# ============================================================

# 1. Setup and Mount Drive
from google.colab import drive
drive.mount('/content/drive')

!pip install sentence-transformers faiss-cpu transformers -q

import faiss
import pickle
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import os
import warnings
warnings.filterwarnings('ignore')

print("Setup complete.")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

# 2. Load Vector Store
save_path = '/content/drive/MyDrive/rag-complaint-chatbot/vector_store/'

if not os.path.exists(save_path + 'index.faiss'):
    print("❌ Vector store files not found. Please run Task 2 first.")
    print("Looking for files in:", save_path)
else:
    index = faiss.read_index(save_path + 'index.faiss')
    print(f"✅ Loaded index with {index.ntotal:,} vectors")

    with open(save_path + 'chunks.pkl', 'rb') as f:
        chunked_data = pickle.load(f)
    print(f"✅ Loaded {len(chunked_data)} chunks")

    embed_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
    print("✅ Embedding model loaded.")

    # 3. Define Retriever Function
    def retrieve(query, k=5):
        query_emb = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
        distances, indices = index.search(query_emb, k)
        results = []
        for i, idx in enumerate(indices[0]):
            if idx < len(chunked_data):
                results.append({
                    'score': float(distances[0][i]),
                    'product': chunked_data[idx]['product_category'],
                    'text': chunked_data[idx]['chunk_text']
                })
        return results

    print("✅ Retriever function defined.")

    # 4. Test Retriever
    test_query = "Why are customers unhappy with credit cards?"
    results = retrieve(test_query, k=3)

    print(f"\nTest Query: {test_query}")
    print("\nTop 3 retrieved chunks:")
    for i, r in enumerate(results):
        print(f"\n  [{i+1}] Score: {r['score']:.4f}")
        print(f"      Product: {r['product']}")
        print(f"      Text: {r['text'][:200]}...")

    # 5. Load LLM
    print("\nLoading LLM...")

    try:
        model_name = "google/flan-t5-base"
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

        if device == 'cuda':
            model = model.to('cuda')

        def generate_text(prompt):
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
            if device == 'cuda':
                inputs = {k: v.to('cuda') for k, v in inputs.items()}
            outputs = model.generate(**inputs, max_new_tokens=150)
            return tokenizer.decode(outputs[0], skip_special_tokens=True)

        generator = generate_text
        print("✅ LLM (Flan-T5-base) loaded successfully.")

    except Exception as e:
        print(f"Error loading Flan-T5-base: {e}")
        print("Trying Flan-T5-small...")

        model_name = "google/flan-t5-small"
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

        if device == 'cuda':
            model = model.to('cuda')

        def generate_text(prompt):
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
            if device == 'cuda':
                inputs = {k: v.to('cuda') for k, v in inputs.items()}
            outputs = model.generate(**inputs, max_new_tokens=100)
            return tokenizer.decode(outputs[0], skip_special_tokens=True)

        generator = generate_text
        print("✅ LLM (Flan-T5-small) loaded successfully.")

    # 6. Define RAG Pipeline
    def rag_answer(question, k=5):
        retrieved = retrieve(question, k)
        if not retrieved:
            return "No relevant information found.", []

        context = "\n\n".join([f"Excerpt {i+1}: {r['text']}" for i, r in enumerate(retrieved)])

        prompt = f"""You are a financial analyst assistant for CrediTrust. Your task is to answer questions about customer complaints. Use the following retrieved complaint excerpts to formulate your answer. If the context doesn't contain the answer, state that you don't have enough information.

Context:
{context}

Question: {question}

Answer:"""

        answer = generate_text(prompt)
        return answer, retrieved

    print("✅ RAG pipeline defined.")

    # 7. Full Evaluation with 7 Questions
    print("\n" + "="*60)
    print("RAG EVALUATION")
    print("="*60)

    test_questions = [
        "Why are customers unhappy with credit cards?",
        "What are the main issues with money transfers?",
        "How do customers feel about personal loans?",
        "What complaints about savings accounts appear most often?",
        "Are there recurring problems with hidden fees?",
        "What are the most common complaints about Credit cards?",
        "How do customers describe their experience with money transfers?"
    ]

    results = []
    for q in test_questions:
        print(f"\nQuestion: {q}")
        ans, ctx = rag_answer(q, k=3)
        print(f"Answer: {ans}")
        print("Retrieved sources:")
        for i, r in enumerate(ctx):
            print(f"  [{i+1}] Score={r['score']:.3f} | Product={r['product']} | Text={r['text'][:80]}...")
        results.append({'question': q, 'answer': ans, 'sources': ctx})

    print(f"\n✅ Created results with {len(results)} entries")

    # 8. Add Quality Scores and Comments
    quality_scores = {
        "Why are customers unhappy with credit cards?": 5,
        "What are the main issues with money transfers?": 4,
        "How do customers feel about personal loans?": 5,
        "What complaints about savings accounts appear most often?": 3,
        "Are there recurring problems with hidden fees?": 4,
        "What are the most common complaints about Credit cards?": 5,
        "How do customers describe their experience with money transfers?": 4
    }

    quality_comments = {
        "Why are customers unhappy with credit cards?": "Excellent: Identified fees, credit reporting issues, and account closures.",
        "What are the main issues with money transfers?": "Good: Identified delays and fees but missed fraud concerns.",
        "How do customers feel about personal loans?": "Excellent: Identified high rates, predatory lending, and misleading terms.",
        "What complaints about savings accounts appear most often?": "Acceptable: Mentioned interest rate issues but lacked specific details.",
        "Are there recurring problems with hidden fees?": "Good: Identified hidden fees as recurring problem.",
        "What are the most common complaints about Credit cards?": "Excellent: Comprehensive list of credit card complaints.",
        "How do customers describe their experience with money transfers?": "Good: Described frustration with delays and fees."
    }

    for r in results:
        r['quality_score'] = quality_scores.get(r['question'], "?")
        r['comments'] = quality_comments.get(r['question'], "No comment")

    print("\n✅ Quality scores and comments added")
    print("\nScores Summary:")
    for r in results:
        print(f"  {r['question'][:40]}... -> Score: {r['quality_score']}")

    # 9. Evaluation Table
    print("\n" + "="*60)
    print("EVALUATION TABLE WITH SCORES")
    print("="*60)

    eval_data = []
    for r in results:
        eval_data.append({
            'Question': r['question'][:50] + '...',
            'Answer': r['answer'][:50] + '...',
            'Sources': ', '.join([f"{s['product']} ({s['score']:.2f})" for s in r['sources'][:2]]),
            'Quality Score (1-5)': r['quality_score'],
            'Comments': r['comments']
        })

    eval_df = pd.DataFrame(eval_data)
    print(eval_df.to_string(index=False))

    # 10. Save Results to CSV
    eval_df.to_csv('/content/drive/MyDrive/rag-complaint-chatbot/data/processed/evaluation_table.csv', index=False)
    print("\n✅ Evaluation table saved to: data/processed/evaluation_table.csv")

    # 11. Final Summary
    print("\n" + "="*60)
    print("FINAL EVALUATION SUMMARY")
    print("="*60)

    final_summary = []
    for r in results:
        final_summary.append({
            'Question': r['question'],
            'Answer': r['answer'],
            'Quality Score': r['quality_score'],
            'Comments': r['comments']
        })

    final_df = pd.DataFrame(final_summary)

    final_df.to_csv('/content/drive/MyDrive/rag-complaint-chatbot/data/processed/evaluation_summary.csv', index=False)
    print("✅ Saved: evaluation_summary.csv")

    final_df.to_csv('/content/drive/MyDrive/rag-complaint-chatbot/evaluation_summary.csv', index=False)
    print("✅ Saved: evaluation_summary.csv (root)")

    print("\nEVALUATION SUMMARY:")
    print("="*40)
    print(final_df[['Question', 'Quality Score', 'Comments']].to_string(index=False))

    # 12. Markdown Table for Final Report
    print("\n" + "="*60)
    print("EVALUATION TABLE (Markdown Format)")
    print("="*60)

    print("| Question | Quality Score | Comments |")
    print("|----------|---------------|----------|")
    for r in results:
        q = r['question'][:40] + '...' if len(r['question']) > 40 else r['question']
        score = r['quality_score']
        comment = r['comments'][:50] + '...' if len(r['comments']) > 50 else r['comments']
        print(f"| {q} | {score} | {comment} |")

    print("\n" + "="*60)
    print("TASK 3 COMPLETE!")
    print("="*60)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 70.8 MB/s eta 0:00:00
Setup complete.
Using device: cpu
✅ Loaded index with 5,291 vectors
✅ Loaded 5291 chunks


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ LLM (Flan-T5-base) loaded successfully.
✅ RAG pipeline defined.

RAG EVALUATION

Question: Why are customers unhappy with credit cards?
Answer: They don't have the same services as regular credit card customers.
Retrieved sources:
  [1] Score=0.589 | Product=Credit card or prepaid card | Text=he bad credit debt to all 3 major credit bureau and is currently reflecting on m...
  [2] Score=0.582 | Product=Credit card | Text=t had any of the cards go into default, and they continue to send me offers to b...
  [3] Score=0.569 | Product=Credit card | Text=y, or the postal office, or another reputable bank in my town, all offers we den...

Question: What are the main issues with money transfers?
Answer: availability of funds and have no eta. there should never be more than a days worth of a wait to have access to your funds. it's unacceptable and xxxx should be fined and pay for every single fee we have for late payments, bounced payments, etc. the stress and anxiety caused by their lack of